# 01 — Colab: Open-LLM Inference

Run this notebook on **Google Colab** with **GPU enabled**. It extracts the prepared input bundle from Google Drive (`political-bias-lab/transfer`), performs all LLM inference, saves resumable Parquet checkpoints, cross-judges Phase 4 with the other open model, and saves the output bundle back to Google Drive.

In [ ]:
!pip install -U "bitsandbytes>=0.46.1" accelerate

In [ ]:
from google.colab import drive
import bitsandbytes
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
REPO_URL = "https://github.com/MichealSK/political-bias-lab.git"
PROFILE = "smoke"  # must match Colab preparation profile
DRIVE_ROOT = "/content/drive/MyDrive/political-bias-lab"
TRANSFER_DIR = f"{DRIVE_ROOT}/transfer"
REPO_DIR = "/content/political-bias-lab"

In [ ]:
import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

Working directory: /content/political-bias-lab
Git revision: 3b277b61c140eb4fa23fc0c87c1f6d24f4376a7c


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])

0

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a Colab GPU accelerator before running inference."
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

GPU: Tesla T4
CUDA: 12.8


## Unpack prepared Google Drive inputs into working directory

In [ ]:
import zipfile
from pathlib import Path

bundle_path = Path(TRANSFER_DIR) / f"kaggle_input_{PROFILE}.zip"
assert bundle_path.exists(), f"Input bundle not found at {bundle_path}. Run notebook 00 first."

with zipfile.ZipFile(bundle_path, "r") as zip_ref:
    zip_ref.extractall(REPO_DIR)

print("Extracted bundle contents to:", REPO_DIR)
print(sorted(p.name for p in (Path(REPO_DIR)/"prepared").glob("*.parquet")))

Extracted bundle contents to: /content/political-bias-lab
['bias_exemplars.parquet', 'entities.parquet', 'entity_pairs.parquet', 'evasion_items.parquet', 'persona_items.parquet', 'phase1_sample.parquet', 'pluralism_cases.parquet', 'sentiment_templates.parquet']


In [ ]:
from src.config import load_config
from src.io_utils import read_json
from src.reproducibility import save_run_manifest
from pathlib import Path
import subprocess

cfg = load_config(Path(REPO_DIR)/"config/default.yaml", Path(REPO_DIR)/f"config/{PROFILE}.yaml")
prep = read_json(Path(REPO_DIR)/"results/manifests/data_preparation.json", {})
if prep:
    assert prep.get("profile") == PROFILE, f"Prepared data profile {prep.get('profile')} does not match profile {PROFILE}"

prepare_manifest = read_json(Path(REPO_DIR)/"results/manifests/colab_prepare_manifest.json", {})
current_git = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if prepare_manifest.get("git_revision"):
    assert prepare_manifest["git_revision"] == current_git, (
        "Git revision mismatch. Check out the exact revision used in Colab preparation: ",
        prepare_manifest["git_revision"], current_git
    )

save_run_manifest(Path(REPO_DIR)/"results/manifests/colab_inference_manifest.json", config=cfg, root=REPO_DIR, extra={"stage":"colab_inference"})
print("Profile:", PROFILE)

Profile: smoke


## Run evaluated models sequentially

The first model is unloaded before the second model loads, which minimizes VRAM use. Successful rows are checkpointed; failed rows can be retried on a rerun.

In [ ]:
from src.pipeline import run_one_model
from src.io_utils import write_json
from pathlib import Path

model_manifests = []
for model_cfg in cfg["models"]["evaluated"]:
    print("\n=== Running", model_cfg["id"], "===")
    model_manifests.append(run_one_model(cfg, model_cfg, root=REPO_DIR))
write_json(model_manifests, Path(REPO_DIR)/"results/manifests/evaluated_models.json")
model_manifests


=== Running HuggingFaceTB/SmolLM2-1.7B-Instruct ===


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

...

Phase 1 | qwen2.5-1.5b:   0%|          | 0/32 [00:00<?, ?it/s]

Phase 2 | qwen2.5-1.5b:   0%|          | 0/18 [00:00<?, ?it/s]

Phase 3 | qwen2.5-1.5b:   0%|          | 0/12 [00:00<?, ?it/s]

Phase 4 generation | qwen2.5-1.5b:   0%|          | 0/8 [00:00<?, ?it/s]

[{'model_id': 'HuggingFaceTB/SmolLM2-1.7B-Instruct',
  'short_name': 'smollm2-1.7b',
  'requested_revision': None,
  'resolved_revision': '31b70e2e869a7173562077fd711b654946d38674',
  'quantized_4bit': True,
  'device_type': 'cuda',
  'gpu_name': 'Tesla T4',
  'max_context_tokens': 4096},
 {'model_id': 'Qwen/Qwen2.5-1.5B-Instruct',
  'short_name': 'qwen2.5-1.5b',
  'requested_revision': None,
  'resolved_revision': '989aa7980e4cf806f80c7fef2b1adb7bc71aa306',
  'quantized_4bit': True,
  'device_type': 'cuda',
  'gpu_name': 'Tesla T4',
  'max_context_tokens': 4096}]

## Cross-model Phase-4 judging

Each default evaluated model judges only the other model's outputs, never its own.

In [ ]:
from src.pipeline import run_cross_judging
from src.io_utils import write_json
judge_manifests = run_cross_judging(cfg, root=REPO_DIR)
write_json(judge_manifests, Path(REPO_DIR)/"results/manifests/judge_models.json")
judge_manifests

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Phase 4 judge | smollm2-1.7b:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Phase 4 judge | qwen2.5-1.5b:   0%|          | 0/8 [00:00<?, ?it/s]

[{'model_id': 'HuggingFaceTB/SmolLM2-1.7B-Instruct',
  'short_name': 'smollm2-1.7b',
  'requested_revision': None,
  'resolved_revision': '31b70e2e869a7173562077fd711b654946d38674',
  'quantized_4bit': True,
  'device_type': 'cuda',
  'gpu_name': 'Tesla T4',
  'max_context_tokens': 4096},
 {'model_id': 'Qwen/Qwen2.5-1.5B-Instruct',
  'short_name': 'qwen2.5-1.5b',
  'requested_revision': None,
  'resolved_revision': '989aa7980e4cf806f80c7fef2b1adb7bc71aa306',
  'quantized_4bit': True,
  'device_type': 'cuda',
  'gpu_name': 'Tesla T4',
  'max_context_tokens': 4096}]

## Sanity-check failures before export

In [ ]:
import pandas as pd
from pathlib import Path
raw_dir = Path(REPO_DIR)/"results/raw"
for p in sorted(raw_dir.glob("*.parquet")):
    df = pd.read_parquet(p)
    failures = int(df["error"].notna().sum()) if "error" in df.columns else 0
    print(p.name, "rows=", len(df), "failures=", failures)

phase1.parquet rows= 64 failures= 0
phase2.parquet rows= 36 failures= 0
phase3.parquet rows= 24 failures= 0
phase4_generations.parquet rows= 16 failures= 0
phase4_judgments.parquet rows= 16 failures= 0


## Export output bundle to Google Drive

In [ ]:
from src.transfer import make_kaggle_output_bundle
from pathlib import Path
import shutil

local_bundle = Path(REPO_DIR)/"transfer"/f"colab_results_{PROFILE}.zip"
make_kaggle_output_bundle(REPO_DIR, local_bundle)

drive_transfer = Path(DRIVE_ROOT)/"transfer"
drive_transfer.mkdir(parents=True, exist_ok=True)
drive_bundle = drive_transfer / local_bundle.name
shutil.copy2(local_bundle, drive_bundle)
print("Saved output bundle to Google Drive:", drive_bundle)

Saved output bundle to Google Drive: /content/drive/MyDrive/political-bias-lab/transfer/colab_results_smoke.zip
